In [2]:
!pip install pillow

/bin/bash: line 1: /home/archelephant/Desktop/Projects/Ollama/.venv/bin/pip: cannot execute: required file not found


In [1]:
import requests
import json
import time
import sys
from pathlib import Path
from typing import Optional, Dict, Any, Generator
import logging
from pathlib import Path
from PIL import Image
import io

# Настройка логирования для отладки
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Конфигурация Ollama-сервера
OLLAMA_BASE_URL = "https://ollama.lourie.info"
#Read credentials from file
with open('Ollama_Creds.json', 'r') as f:
    creds = json.load(f)
    f.close()
AUTH = (creds['User'], creds['Password'])
MODEL_NAME = 'gemma3:4b'  # Используем мультимодальную модель с поддержкой изображений

# ---------- Helper: encode image to base64 ----------
def encode_image(image_path: Path) -> str:
    """
    Reads an image file and returns its base64 string (without data URI prefix).
    Supports common formats: JPEG, PNG, etc.
    """
    with open(image_path, "rb") as img_file:
        return base64.b64encode(img_file.read()).decode('utf-8')

# ---------- Send image + prompt (non‑streaming) ----------
def ask_model_with_image(prompt: str, image_path: Path, model: str = MODEL_NAME,
                         stream: bool = False) -> dict:
    """
    Sends a prompt and an image to the Ollama /api/generate endpoint.
    Returns the full JSON response (or prints tokens if streaming).
    """
    url = f"{OLLAMA_BASE_URL}/api/generate"
    
    # Prepare payload
    image_b64 = encode_image(image_path)
    payload = {
        "model": model,
        "prompt": prompt,
        "images": [image_b64],   # list of base64 strings
        "stream": stream,
        "options": {
            "temperature": 0.7,   # optional
            "top_p": 0.9
        }
    }
    
    headers = {"Content-Type": "application/json"}
    
    if not stream:
        # Non‑streaming: wait for complete response
        response = requests.post(
            url,
            auth=AUTH,
            headers=headers,
            json=payload,
            timeout=(10, 120)   # connect + read timeout
        )
        response.raise_for_status()
        return response.json()
    else:
        # Streaming: yield each chunk
        with requests.post(
            url,
            auth=AUTH,
            headers=headers,
            json=payload,
            stream=True,
            timeout=(10, 120)
        ) as r:
            r.raise_for_status()
            for line in r.iter_lines():
                if line:
                    chunk = json.loads(line.decode('utf-8'))
                    yield chunk

# Параметры запроса
REQUEST_TIMEOUT = (10, 300)  # (connect timeout, read timeout) в секундах
MAX_RETRIES = 5
RETRY_BACKOFF_FACTOR = 2  # множитель для задержки между попытками
STREAM_CHUNK_SIZE = 1024

def check_ollama_health():
    """
    Verify that the ollama server is reachable and authentication works.
    Uses the /api/tags endpoint which lists available models.
    Returns True if healthy, False otherwise.
    """
    url = f"{OLLAMA_BASE_URL}/api/tags"
    try:
        response = requests.get(url, auth=AUTH, timeout=10)
        response.raise_for_status()
        data = response.json()
        models = [m["name"] for m in data.get("models", [])]
        print(f"Ollama server OK. Available models: {models}")
        return True
    except Exception as e:
        print(f"Ollama health check failed: {e}")
        return False
    
#Test the oLLAMA check method:
check_ollama_health()

ModuleNotFoundError: No module named 'PIL'